# Lab 3 - Power Analysis and Code Flow

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *Now that you've been introduced to the ChipWhisperer platform, we'll be using it to learn more about how the power consumed by a microcontroller varies based on what operations it is performing and what instructions it is executing.*

**LEARNING OUTCOMES:**

* Capturing a power trace with ChipWhisperer
* Modifying target firmware
* Making connections between various simple instructions and power traces

## Prerequisites

Hold up! Before you continue, check you've done the following:

* ☑ Run through [Lab 2 - ChipWhisperer's Python API](../Lab%202%20-%20ChipWhisperer's%20Python%20API/Lab.ipynb).

## Setup

Let's start off by connecting to our ChipWhisperer-Husky and running the default setup. To do this, you just need to rerun the first couple of commands from the last lab.

If you get USB errors in the above block, you may have left your scope connected in the previous lab. In general, you should always disconnect your scope at the end of the lab, but if you forget, unplugging+replugging should solve the issue.

This lab is fairly simple and open ended - the goal here is to take the firmware from the previous lab, modify it, and see how the power traces change. Let's first confirm that we can still build our firmware:

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-base
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE

and program our target:

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

Then confirm that we can still capture a trace. We're going to use the `cw.capture_trace()` function that I mentioned in the last lab.

**Make a function called capture_trace() that sends 16 `0x00` bytes over SimpleSerial and returns the resulting power trace:**

In [ ]:
wave = capture_trace()
assert len(wave) == 5000, "Did you remember to return the power trace in your function?"

## "Empty" Trace

Now that we've built and captured trace, let's take a look at our firmware again. We'll be looking at the `get_pt()` function again. Go open up `chipwhisperer/firmware/mcu/simpleserial-base/simpleserial-base.c` and take a look at this function.

As you can see, there's not a lot happening in this function - it's got a `trigger_high()` call, then a `trigger_low()` call, then it sends a SimpleSerial packet back to us, then finally it returns. Before we move on I do want to clarify a few things:

1. `trigger_high()` is the starting point for our capture: nothing before this call will be in our power trace, while things afterwards will be.
2. How much is captured afterwards is solely determined by `scope.adc.samples`. `trigger_low()` has no effect on the capture.

We've now captured a trace with 5000 samples. Let's see what that looks like when we plot:

In [ ]:
cw.plot(wave)

You'll probably see a few spikes, then a bunch of oscillations. What we're seeing is the following:

1. The target jumping back out of the `trigger_high()` call, then into the `trigger_low()` call.
2. Oscillations from the target sending serial messages back to us.

There isn't really a lot interesting here, but this is a good reference for when we actually start adding things to this function.

## Simple Instructions

To start off, let's insert some simple instructions to see if they're visible in the power trace. Try inserting the following code, which will execute 20 multiplies. It's important to mark `A` as `volatile` here to prevent the compiler from optimizing all these instructions out:

```C
	volatile long int A = 0x2BAA;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;

	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
```

Then rebuild, capture a new trace, and plot it:

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-base
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

In [ ]:
wave2 = capture_trace()

cw.plot(wave2)

To make things a little easier to see, let's plot our original trace and our new trace on the same plot. We can simply subtract the midway point from the plot to shift it up or down, which should make things a little easier to see:

In [ ]:
cw.plot(wave) * cw.plot(wave2 - 2500)

## Simple Instruction Loop

You might think that it's strange that we didn't simply use a loop instead of copying the same thing out 20 times. Rectify this by replacing these instructions with a loop. **Make sure to make the loop variable volatile as well.**

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-base
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

In [ ]:
wave3 = capture_trace()
cw.plot(wave3) * cw.plot(wave2 - 2500)

Are you surprised by how different the loop is from just repeatedly performing operations? If you're comfortable with assembly, you can check the `.lss` file in the `simpleserial-base-lab2` directory. Otherwise, here's a C level overview of what (depending on how your compiler lays things out) the loop acutally looks like:

```C
i=0;
start:
    if (i > 20) {
        goto end;
    }    
    A *= 2;
    i += 1;
    goto start;
end:
    trigger_low();
    //...
```

As you can see, the microcontroller has to do a lot more than simply repeating our instruction. As such, it's something a compiler will often "unroll" (basically turn it back into our original repeated multiplies). In our case, since `i` was marked as `volatile`, the compiler avoided making this optimization. Unrolling loops also increases code size, so the compiler might also avoid unrolling when optimizing for code size.

## Expensive Instructions

Up to now, all the instructions we've used (besides branches in the loop) have executed in a single clock cycle. Instead, let's try an instruction that takes multiple clock cycles to execute. For Arm devices like our SAM4S, divide is an easy one. Try changing the multiply to a divide by 2 instead.

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-base
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

In [ ]:
wave4 = capture_trace()
cw.plot(wave4) * cw.plot(wave3 - 2500)

You can probably see a similar pattern to the faster instruction, but each go through the loop takes longer. We would also expect that longer instructions would also consume more power. Can you see this in your plot?

**HINT: In order to measure current, ChipWhisperer measures voltage drop across a shunt resistor. This means that the power trace is actually inverted (aka large negative swings are areas of higher power consumption).**

## Conclusions & Next Steps

By now you should be reasonably convinced that we can gain some understanding of what a microcontroller is doing by looking at its power consumption. Going forward, we'll be focusing on more objective measurements, but what we did in this lab is still very valuable. AES implementations, for example, will often have a very distinct shape. You can use this to help identify what an unknown device is doing at different times.

Of course, there's a lot more you can do with this lab. Try changing the division to different values. Does the power trace look any different?

In [ ]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>